### Libraries

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

### Step 1: Load all evaluation results

In [2]:
PARAM_DIR = '../models/Project_Parameter_Files/Evalution_result'

cls_val_results     = pd.read_csv(f'{PARAM_DIR}/cls_val_results.csv',index_col=0)
cls_test_results    = pd.read_csv(f'{PARAM_DIR}/cls_test_results.csv',index_col=0)
reg_val_results     = pd.read_csv(f'{PARAM_DIR}/reg_val_results.csv',index_col=0)
reg_test_results    = pd.read_csv(f'{PARAM_DIR}/reg_test_results.csv',index_col=0)
cluster_val_results = pd.read_csv(f'{PARAM_DIR}/cluster_val_results.csv',index_col=0)
cluster_test_results = pd.read_csv(f'{PARAM_DIR}/cluster_test_results.csv',index_col=0)

print("All evaluation results loaded")
print(f"\nClassification models : {cls_test_results.index.tolist()}")
print(f"Regression models     : {reg_test_results.index.tolist()}")
print(f"Clustering models     : {cluster_test_results.index.tolist()}")

All evaluation results loaded

Classification models : ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM']
Regression models     : ['LinearRegression', 'RandomForest', 'XGBoost', 'LightGBM']
Clustering models     : ['KMeans', 'GMM']


### Step 2: Classification — final comparison table

In [3]:
print("CLASSIFICATION — FINAL COMPARISON")

# Combine val and test
cls_comparison = pd.DataFrame({
    'Val F1':        cls_val_results['f1'],
    'Test F1':       cls_test_results['f1'],
    'Val AUC':       cls_val_results['roc_auc'],
    'Test AUC':      cls_test_results['roc_auc'],
    'Val Accuracy':  cls_val_results['accuracy'],
    'Test Accuracy': cls_test_results['accuracy'],
    'Val Precision': cls_val_results['precision'],
    'Test Precision':cls_test_results['precision'],
    'Val Recall':    cls_val_results['recall'],
    'Test Recall':   cls_test_results['recall'],
}).round(4)

print(cls_comparison.to_string())

print("\n Best Model Per Metric")
for col in cls_comparison.columns:
    best = cls_comparison[col].idxmax()
    print(f"  {col:<20} : {best} ({cls_comparison.loc[best, col]:.4f})")

CLASSIFICATION — FINAL COMPARISON
                    Val F1  Test F1  Val AUC  Test AUC  Val Accuracy  Test Accuracy  Val Precision  Test Precision  Val Recall  Test Recall
LogisticRegression  0.6566   0.6672   0.8630    0.8735        0.8412         0.8384         0.6214          0.6055      0.6961       0.7430
RandomForest        0.7847   0.8040   0.9259    0.9307        0.9218         0.9272         0.9831          0.9733      0.6529       0.6848
XGBoost             0.8197   0.8321   0.9399    0.9458        0.9313         0.9341         0.9574          0.9366      0.7167       0.7486
LightGBM            0.8245   0.8390   0.9414    0.9471        0.9309         0.9358         0.9233          0.9253      0.7448       0.7674

 Best Model Per Metric
  Val F1               : LightGBM (0.8245)
  Test F1              : LightGBM (0.8390)
  Val AUC              : LightGBM (0.9414)
  Test AUC             : LightGBM (0.9471)
  Val Accuracy         : XGBoost (0.9313)
  Test Accuracy        : Lig

In [4]:
# Classification decision is clear — LightGBM

### Step 3: Regression — final comparison table

In [5]:
print("REGRESSION — FINAL COMPARISON")

reg_comparison = pd.DataFrame({
    'Val R2':    reg_val_results['r2'],
    'Test R2':   reg_test_results['r2'],
    'Val RMSE':  reg_val_results['rmse'],
    'Test RMSE': reg_test_results['rmse'],
    'Val MAE':   reg_val_results['mae'],
    'Test MAE':  reg_test_results['mae'],
}).round(4)

print(reg_comparison.to_string())

# for RMSE and MAE lower is better so use idxmin
print("\n Best Model Per Metric")
for col in reg_comparison.columns:
    if 'R2' in col:
        best = reg_comparison[col].idxmax()
        print(f"  {col:<15} : {best} ({reg_comparison.loc[best, col]:.4f}) ↑ higher better")
    else:
        best = reg_comparison[col].idxmin()
        print(f"  {col:<15} : {best} ({reg_comparison.loc[best, col]:.4f}) ↓ lower better")

REGRESSION — FINAL COMPARISON
                  Val R2  Test R2  Val RMSE  Test RMSE  Val MAE  Test MAE
LinearRegression  0.8826   0.8772    1.1476     1.1318   0.9045    0.8869
RandomForest      0.9130   0.9077    0.9878     0.9811   0.7849    0.7697
XGBoost           0.9127   0.9071    0.9899     0.9844   0.7897    0.7707
LightGBM          0.9115   0.9066    0.9967     0.9872   0.7942    0.7722

 Best Model Per Metric
  Val R2          : RandomForest (0.9130) ↑ higher better
  Test R2         : RandomForest (0.9077) ↑ higher better
  Val RMSE        : RandomForest (0.9878) ↓ lower better
  Test RMSE       : RandomForest (0.9811) ↓ lower better
  Val MAE         : RandomForest (0.7849) ↓ lower better
  Test MAE        : RandomForest (0.7697) ↓ lower better


In [6]:
# Regression decision is clear — RandomForest

### Step 4: Clustering Final Comparison Table

In [7]:
print("CLUSTERING — FINAL COMPARISON")

cluster_comparison = pd.DataFrame({
    'Val Silhouette':        cluster_val_results['silhouette'],
    'Test Silhouette':       cluster_test_results['silhouette'],
    'Val Davies Bouldin':    cluster_val_results['davies_bouldin'],
    'Test Davies Bouldin':   cluster_test_results['davies_bouldin'],
    'Val Calinski Harabasz': cluster_val_results['calinski_harabasz'],
    'Test Calinski Harabasz':cluster_test_results['calinski_harabasz'],
}).round(4)

print(cluster_comparison.to_string())

# Silhouette + Calinski — higher is better
# Davies Bouldin — lower is better
print("\n Best Model Per Metric")
for col in cluster_comparison.columns:
    if 'Davies' in col:
        best = cluster_comparison[col].idxmin()
        print(f"  {col:<30} : {best} ({cluster_comparison.loc[best, col]:.4f}) ↓ lower better")
    else:
        best = cluster_comparison[col].idxmax()
        print(f"  {col:<30} : {best} ({cluster_comparison.loc[best, col]:.4f}) ↑ higher better")

CLUSTERING — FINAL COMPARISON
        Val Silhouette  Test Silhouette  Val Davies Bouldin  Test Davies Bouldin  Val Calinski Harabasz  Test Calinski Harabasz
KMeans          0.1584           0.1508              2.8406               2.8335               236.4955                258.2045
GMM             0.2287           0.2196              4.2214               4.3057               115.7651                118.0826

 Best Model Per Metric
  Val Silhouette                 : GMM (0.2287) ↑ higher better
  Test Silhouette                : GMM (0.2196) ↑ higher better
  Val Davies Bouldin             : KMeans (2.8406) ↓ lower better
  Test Davies Bouldin            : KMeans (2.8335) ↓ lower better
  Val Calinski Harabasz          : KMeans (236.4955) ↑ higher better
  Test Calinski Harabasz         : KMeans (258.2045) ↑ higher better


### Step 5: Final model decision + justification

In [8]:
print("FINAL MODEL SELECTION — ALL TASKS")

# CLASSIFICATION
print("\n--- Classification ---")
print("""
Candidates : LightGBM vs XGBoost

LightGBM:
  Test F1        = 0.8390 
  Test AUC       = 0.9471 
  Test Recall    = 0.7674  — catches most defaulters
  Test Accuracy  = 0.9358 
  Calibration    = Good

XGBoost:
  Test F1        = 0.8321
  Test AUC       = 0.9458
  Test Precision = 0.9366  — fewer false alarms
  Calibration    = Best

Decision: LightGBM
Reason  : Wins 7/10 metrics. Higher recall catches
          more defaulters — critical for loan risk.
          Small calibration difference not worth
          sacrificing F1 and recall.
""")

# REGRESSION
print("\n--- Regression ---")
print("""
Candidates : RandomForest vs XGBoost vs LightGBM

RandomForest:
  Test R2   = 0.9077 
  Test RMSE = 0.9811 
  Test MAE  = 0.7697 
  Wins all 6/6 metrics

Decision: RandomForest
Reason  : Wins every single metric. No debate.
""")

# CLUSTERING
print("\n--- Clustering ---")
print("""
Candidates : GMM vs KMeans

GMM:
  Test Silhouette = 0.2196 
  Soft probabilities — can say 70% High Value
  Better cluster separation

KMeans:
  Test Davies Bouldin    = 2.8335 
  Test Calinski Harabasz = 258.20 
  Tighter more compact clusters

Decision: GMM
Reason  : Silhouette is primary metric for cluster
          quality. Soft probabilities add real
          production value — API can return
          probability of cluster membership.
          KMeans compact clusters are less useful
          than GMM interpretability.
""")

# FINAL SUMMARY
print("FINAL SELECTED MODELS")
print(f"  Classification : cls_LightGBM   (F1=0.8390, AUC=0.9471)")
print(f"  Regression     : reg_RandomForest (R2=0.9077, RMSE=0.9811)")
print(f"  Clustering     : cluster_GMM    (Silhouette=0.2196)")
print(f"  Cls Threshold  : 0.6")

FINAL MODEL SELECTION — ALL TASKS

--- Classification ---

Candidates : LightGBM vs XGBoost

LightGBM:
  Test F1        = 0.8390 
  Test AUC       = 0.9471 
  Test Recall    = 0.7674  — catches most defaulters
  Test Accuracy  = 0.9358 
  Calibration    = Good

XGBoost:
  Test F1        = 0.8321
  Test AUC       = 0.9458
  Test Precision = 0.9366  — fewer false alarms
  Calibration    = Best

Decision: LightGBM
Reason  : Wins 7/10 metrics. Higher recall catches
          more defaulters — critical for loan risk.
          Small calibration difference not worth
          sacrificing F1 and recall.


--- Regression ---

Candidates : RandomForest vs XGBoost vs LightGBM

RandomForest:
  Test R2   = 0.9077 
  Test RMSE = 0.9811 
  Test MAE  = 0.7697 
  Wins all 6/6 metrics

Decision: RandomForest
Reason  : Wins every single metric. No debate.


--- Clustering ---

Candidates : GMM vs KMeans

GMM:
  Test Silhouette = 0.2196 
  Soft probabilities — can say 70% High Value
  Better cluster sepa

### Step 6: Save Final Model Config

In [9]:
# Final model config
final_model_config = {
    'classification': {
        'model_name':  'cls_LightGBM',
        'model_file':  'cls_LightGBM.joblib',
        'threshold':    0.6,
        'metrics': {
            'test_f1':       0.8390,
            'test_auc':      0.9471,
            'test_accuracy': 0.9358,
            'test_recall':   0.7674,
            'test_precision':0.9253
        },
        'reason': 'Wins 7/10 metrics. Higher recall critical for loan risk.'
    },
    'regression': {
        'model_name': 'reg_RandomForest',
        'model_file': 'reg_RandomForest.joblib',
        'metrics': {
            'test_r2':   0.9077,
            'test_rmse': 0.9811,
            'test_mae':  0.7697
        },
        'reason': 'Wins all 6/6 metrics. No debate.'
    },
    'clustering': {
        'model_name': 'cluster_GMM',
        'model_file': 'cluster_GMM.joblib',
        'metrics': {
            'test_silhouette':        0.2196,
            'test_davies_bouldin':    4.3057,
            'test_calinski_harabasz': 118.08
        },
        'cluster_labels': {
            '0': 'High Value Borrower',
            '1': 'Standard Borrower'
        },
        'reason': 'Best silhouette + soft probabilities for production.'
    }
}

# Save
PARAM_DIR = '../models/Project_Parameter_Files'
with open(f'{PARAM_DIR}/final_model_config.json', 'w') as f:
    json.dump(final_model_config, f, indent=4)

print("Final model config saved")
print(f"\nSaved: {PARAM_DIR}/final_model_config.json")
print("\nFinal Selected Models:")
for task, config in final_model_config.items():
    print(f"  {task:<20} : {config['model_name']}")

Final model config saved

Saved: ../models/Project_Parameter_Files/final_model_config.json

Final Selected Models:
  classification       : cls_LightGBM
  regression           : reg_RandomForest
  clustering           : cluster_GMM


### Step 7: Document Report

In [10]:
# see in dics/model_selection_report.md